# AICE Associate 변주 문제 v2 - 이탈, 유지 예측 (분류)

### 시나리오

스타일 중심의 패션 이커머스 회사인 '스타일온(StyleOn)'은 신규 고객 유치 및 기존 고객 유지에 중점을 두고 있습니다. 회사는 최근 3개월간의 구매 행동 데이터를 분석하여, 향후 30일 이내에 재구매를 하지 않을 것으로 예측되는 고객을 사전에 식별하고자 합니다. 이를 통해 맞춤형 리텐션 캠페인을 진행하고 이탈률을 최소화하는 전략을 수립하는 것이 목표입니다.

---

**[유의사항]**
- 답안은 각 문항 아래 표시된 `# (N) 여기에 ...` 칸에 작성하세요.
- **정답/해설은 이 노트북 가장 아래 `## 해설` 섹션에 모아뒀습니다.** 먼저 스스로 풀어본 뒤에 확인하세요.
- 이 노트북은 오리지널 창작 문제이며, 실제 AICE 샘플문항 원문을 복제하지 않습니다.
- 데이터 로더: `read_csv` / 기본모델: `DecisionTreeClassifier` / 비교모델: `RandomForestClassifier` / 스케일러: `StandardScaler`

**[데이터 컬럼 설명]**

- Churned : 이탈, 유지
- Total_Purchase_Amount : 고객이 특정 기간 동안 지출한 총 금액 (원)
- Customer_Segment : 고객 세그먼트 (프리미엄,일반,신규,VIP)
- Region_Code : 지역 코드 (KR,US,JP,EU 등 5~8개 코드값) (dim.csv 와 병합 키)
- Days_Since_Last_Purchase : 피처 컬럼
- Average_Order_Value : 피처 컬럼
- Return_Rate_Percentage : 피처 컬럼
- Frequency_of_Login : 피처 컬럼
- Customer_ID : 식별자(모델링에 불필요)
- (병합 후) dim_value : 지역별 평균 구매 빈도

## 0. 데이터 준비

다음 문항을 풀기 전에 아래 코드를 실행하세요 (문제 데이터 2개 테이블을 생성합니다).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


def synthesize_tables(seed=2006, n_rows=600):
    rng = np.random.default_rng(seed)
    n = n_rows

    base = rng.normal(loc=50, scale=15, size=n)
    outlier_idx = rng.choice(n, size=max(3, n // 50), replace=False)
    base[outlier_idx] += rng.choice([1, -1], size=len(outlier_idx)) * rng.uniform(80, 150, size=len(outlier_idx))

    main_df = pd.DataFrame({'Total_Purchase_Amount': base.round(2)})

    cats = [f"Cat{i+1}" for i in range(rng.integers(3, 5))]
    main_df['Customer_Segment'] = rng.choice(cats, size=n)

    n_regions = rng.integers(5, 9)
    regions = [f"R{i+1:02d}" for i in range(n_regions)]
    main_df['Region_Code'] = rng.choice(regions, size=n)

    for col in ['Days_Since_Last_Purchase', 'Average_Order_Value', 'Return_Rate_Percentage', 'Frequency_of_Login']:
        main_df[col] = rng.normal(0, 1, size=n).round(2)

    main_df['Customer_ID'] = [f"ID{i:05d}" for i in range(n)]

    z = (base - base.mean()) / (base.std() + 1e-9)
    classes = ['이탈', '유지']
    if '분류' == "분류":
        if False and len(classes) >= 3:
            bins = np.quantile(z, [1 / 3, 2 / 3])
            idx = np.digitize(z, bins)
            main_df['Churned'] = [classes[i] for i in idx]
        else:
            prob = 1 / (1 + np.exp(-z))
            labels = (rng.random(n) < prob).astype(int)
            main_df['Churned'] = np.where(labels == 1, classes[0], classes[-1])
    else:
        noise = rng.normal(0, 5, size=n)
        main_df['Churned'] = (base * 1.5 + noise).round(2)

    for col in ['Total_Purchase_Amount'] + ['Days_Since_Last_Purchase', 'Average_Order_Value', 'Return_Rate_Percentage', 'Frequency_of_Login'][:1]:
        na_idx = rng.choice(n, size=int(n * 0.03), replace=False)
        main_df.loc[na_idx, col] = np.nan

    main_df = main_df.sample(frac=1, random_state=seed).reset_index(drop=True)

    dim_df = pd.DataFrame({
        'Region_Code': regions,
        "dim_value": rng.uniform(0.5, 2.0, size=n_regions).round(3),
    })
    return main_df, dim_df


main_df, dim_df = synthesize_tables()
main_df.to_csv("data.csv", index=False)
dim_df.to_csv("dim.csv", index=False)
print("데이터 저장 완료 - data.csv:", main_df.shape, "/ dim.csv:", dim_df.shape)
main_df.head(4)

## <데이터 분석>

### 1. 라이브러리 임포트

pandas, numpy, matplotlib.pyplot, seaborn 을 각각 pd, np, plt, sns 별칭으로 임포트하세요.

In [ ]:
# (1) 여기에 답안코드를 작성하고 실행하세요



### 2. 데이터 로드 (read_csv/read_json)

`data.csv` 를 `pd.read_csv` 로 읽어 **my_data** 에, `dim.csv` 를 `pd.read_csv` 로 읽어 **dim_data** 에 각각 할당하세요.

In [ ]:
# (2) 여기에 답안코드를 작성하고 실행하세요



### 3. 결측치 확인

my_data 의 `Total_Purchase_Amount` 컬럼에 결측치(NaN)가 몇 개 있는지 `isna().sum()` 으로 확인하세요. 몇 개입니까?

In [ ]:
# (3) 여기에 답을 입력하세요 (실행 불필요)



### 4. 데이터 병합 (pd.merge)

`Region_Code` 를 키로 my_data 와 dim_data 를 **left join** 하여 **data_merged** 에 저장하세요.

In [ ]:
# (4) 여기에 답안코드를 작성하고 실행하세요



### 5. 데이터 집계 (groupby)

`Customer_Segment` 별 `Total_Purchase_Amount` 의 평균을 구해 **df_grp** 에 저장하세요.

In [ ]:
# (5) 여기에 답안코드를 작성하고 실행하세요



### 6. 시각화 (subplots)

고객 세그먼트 (프리미엄,일반,신규,VIP)(Customer_Segment) 분포의 countplot 과, 이탈, 유지 별 고객이 특정 기간 동안 지출한 총 금액 (원)(Total_Purchase_Amount) histplot 을 나란히 그리는 코드입니다.
빈칸 **(A)** 에 들어갈, 여러 그래프를 한 번에 그릴 때 쓰는 matplotlib 함수 이름은?

```python
fig, axes = plt.(A)(nrows=1, ncols=2, figsize=(12, 5))
sns.countplot(data=data_merged, x='Customer_Segment', ax=axes[0])
sns.histplot(data=data_merged, x='Total_Purchase_Amount', hue='Churned', ax=axes[1])
plt.show()
```

In [ ]:
# (6) 여기에 답을 입력하세요 (실행 불필요)



### 7. 시각화 2

이탈, 유지 별 고객이 특정 기간 동안 지출한 총 금액 (원)(Total_Purchase_Amount) 분포를 seaborn boxplot 으로 시각화하세요.

In [ ]:
# (7) 여기에 답안코드를 작성하고 실행하세요



## <데이터 전처리>

### 8. 이상치 처리

IQR 기준(K=2.0)을 벗어나는 이상치 행을 제거하고, 식별자 컬럼도 삭제해서 **data_temp** 에 저장하는 코드입니다. 빈칸 **(A)** 에 들어갈, 행을 삭제할 때 쓰는 DataFrame 메서드 이름은?

```python
q1 = data_merged['Total_Purchase_Amount'].quantile(0.25)
q3 = data_merged['Total_Purchase_Amount'].quantile(0.75)
iqr = q3 - q1
lower_fence = q1 - 2.0 * iqr
upper_fence = q3 + 2.0 * iqr
data_temp = data_merged.(A)(data_merged[(data_merged['Total_Purchase_Amount'] > upper_fence) | (data_merged['Total_Purchase_Amount'] < lower_fence)].index)
data_temp = data_temp.drop(columns=['Customer_ID'])
data_temp = data_temp.reset_index(drop=True)
```

In [ ]:
# (8) 여기에 답을 입력하세요 (실행 불필요)



### 9. 결측치 처리

다음은 data_temp 의 결측치를 대표값으로 채우는 코드인데, 실행하면 의도한 것과 다르게 동작합니다. 무엇이 문제인지 서술하거나, 올바르게 고친 코드를 작성하세요.

```python
fill_value = data_temp['Total_Purchase_Amount'].median()
data_na = data_temp.fillna({'Total_Purchase_Amount': fill_value})
if data_na['Days_Since_Last_Purchase'].mean() > 0: 
    data_na['Days_Since_Last_Purchase'] = data_na['Days_Since_Last_Purchase'].fillna(data_na['Days_Since_Last_Purchase'].median())

```

In [ ]:
# (9) 여기에 문제점 설명 또는 수정한 코드를 작성하세요



### 10. 인코딩

범주형 컬럼을 두 그룹으로 나눠 인코딩하세요: `Region_Code` 는 **label_cols** 리스트에 담아 LabelEncoder 로, `Customer_Segment` 는 **dumm_cols** 리스트에 담아 get_dummies 로 인코딩해서 data_preset 에 저장하세요. 타깃 컬럼 `Churned` 도 LabelEncoder 로 정수 인코딩하세요.

In [ ]:
# (10) 여기에 답안코드를 작성하고 실행하세요



### 11. 데이터 분리

Churned 을 y, 나머지를 X 로 삼아 train_test_split 으로 분리하세요.
- test_size=0.2, random_state=7, stratify 옵션을 적용하세요
- 변수명: X_train, X_valid, y_train, y_valid

In [ ]:
# (11) 여기에 답안코드를 작성하고 실행하세요



### 12. 스케일링

StandardScaler 로 훈련/검증 데이터를 스케일링하는 코드인데, 의도한 것과 다르게 동작합니다. 무엇이 문제인지 서술하거나, 올바르게 고친 코드를 작성하세요.

```python
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.transform(X_train)
X_valid_scaled = scaler.fit_transform(X_valid)
```

In [ ]:
# (12) 여기에 문제점 설명 또는 수정한 코드를 작성하세요



### 13. 스케일러 특성

StandardScaler 를 훈련 데이터에 적용하면, 결과 값의 분포는 이론적으로 어떤 특성을 가지게 됩니까?

In [ ]:
# (13) 여기에 답을 입력하세요 (실행 불필요)



## <AI 모델링>

### 14. 머신러닝 기본 (fit-predict)

DecisionTreeClassifier 로 모델을 하나 만들어 학습시키고, 검증데이터에 대한 예측값을 **pred_y** 에 저장하세요. (변수명: model, pred_y)

In [ ]:
# (14) 여기에 답안코드를 작성하고 실행하세요



### 15. GridSearch 모델링

DecisionTreeClassifier 와 RandomForestClassifier 를 GridSearchCV(StratifiedKFold(n_splits=7) 객체, scoring='accuracy')로 탐색하고 학습하세요.
- max_depth 후보: [3, 5, 7]
- RandomForestClassifier 의 n_estimators 후보: [50, 100, 200]
- 변수명: gs_a (베이스 모델), gs_b (비교 모델)

In [ ]:
# (15) 여기에 답안코드를 작성하고 실행하세요



### 16. GridSearch 결과 확인

위 GridSearch에서 RandomForestClassifier 의 n_estimators 후보는 [50, 100, 200] 였습니다. GridSearchCV가 고를 수 있는 값의 후보 중 '가장 큰 값'은 얼마인가요?

In [ ]:
# (16) 여기에 답을 입력하세요 (실행 불필요)



### 17. 변수중요도

RandomForestClassifier 의 변수중요도 Top 15개(정렬 ascending=True)를 뽑아 barh 차트로 시각화하는 코드인데, 의도한 것과 다르게 동작합니다. 무엇이 문제인지 서술하거나, 올바르게 고친 코드를 작성하세요.

```python
fi = pd.DataFrame({'feature': X_train.columns, 'importance': gs_b.best_estimator_.feature_importances_})
fi = fi.sort_values('importance', ascending=True)[:15]
plt.barh(fi['feature'], fi['importance'])
plt.show()
# (이 버전에는 의도한 것과 다른 결과를 내는 부분이 있습니다)
```

In [ ]:
# (17) 여기에 문제점 설명 또는 수정한 코드를 작성하세요



### 18. 성능평가

검증데이터로 gs_a, gs_b 두 모델의 **roc_auc_score** 를 각각 계산해서 a_score, b_score 에 저장하세요 (predict_proba 의 양성 클래스(1) 확률을 사용하세요. 타깃은 인코딩 단계에서 이미 0/1 정수로 변환되어 있습니다).

In [ ]:
# (18) 여기에 답안코드를 작성하고 실행하세요



### 19. 모델 성능 비교

바로 위에서 계산한 a_score 와 b_score 를 비교했을 때, 어느 모델이 더 우수하다고 판단할 수 있습니까? (둘 중 하나를 실행 결과에 따라 답하세요)

In [ ]:
# (19) 여기에 답을 입력하세요 (실행 불필요)



### 20. 딥러닝 설계

다음은 딥러닝 모델을 설계하는 코드입니다. EarlyStopping 에서 '몇 epoch 동안 개선이 없으면 멈출지' 지정하는 파라미터 이름(빈칸 **(A)**)은? (은닉층 활성함수: elu, 출력층: sigmoid/binary_crossentropy, BatchNormalization 포함, ModelCheckpoint 포함)

```python
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

model = Sequential([
    Dense(128, activation='elu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(64, activation='elu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(32, activation='elu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
cb_list = [EarlyStopping(monitor='val_loss', (A)=14, restore_best_weights=True)]
cb_list.append(ModelCheckpoint('best_model.keras', monitor='val_loss', save_best_only=True))
```

In [ ]:
# (20) 여기에 답안코드를 작성하고 실행하세요



### 21. 딥러닝 학습

위에서 설계한 model 을 batch_size=16, epochs=50 으로 학습하고 history 에 저장하세요 (callbacks=cb_list 사용).

In [ ]:
# (21) 여기에 답안코드를 작성하고 실행하세요



### 22. 학습곡선 시각화

history 를 이용해서 학습/검증 **accuracy** 변화를 한 그래프에 시각화하세요 (x축 라벨: epoch, 범례 위치: upper left, 범례 텍스트: train/val).

In [ ]:
# (22) 여기에 답안코드를 작성하고 실행하세요



### 23. 저장된 모델 재사용

ModelCheckpoint 로 저장된 `best_model.keras` 를 `load_model()` 로 다시 불러와서, 검증데이터에 대한 예측값을 **reload_pred** 에 저장하세요.

In [ ]:
# (23) 여기에 답안코드를 작성하고 실행하세요



---
## 해설

스스로 풀어본 뒤 아래에서 확인하세요. 문항 번호가 위 문제 번호와 일치합니다.

### 1번 해설 - 라이브러리 임포트 [코드작성]

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

> import ... as ... 문법으로 널리 쓰이는 관례적 별칭을 지정합니다.

### 2번 해설 - 데이터 로드 (read_csv/read_json) [코드작성]

In [ ]:
my_data = pd.read_csv('data.csv')
dim_data = pd.read_csv('dim.csv')
my_data.head(4)

> pd.read_csv() 로 파일 형식에 맞는 로더를 사용합니다.

### 3번 해설 - 결측치 확인 [결과값예측]

In [ ]:
18

> Series.isna().sum() 은 True(결측치)의 개수를 셉니다.

### 4번 해설 - 데이터 병합 (pd.merge) [코드작성]

In [ ]:
data_merged = pd.merge(my_data, dim_data, on='Region_Code', how='left')
data_merged.head(4)

> pd.merge(left, right, on=키, how='left') 는 왼쪽 테이블 기준으로 오른쪽 테이블을 결합합니다.

### 5번 해설 - 데이터 집계 (groupby) [코드작성]

In [ ]:
df_grp = data_merged.groupby('Customer_Segment')['Total_Purchase_Amount'].mean()
df_grp

> groupby(기준컬럼)[대상컬럼].mean() 형태로 그룹별 집계를 구합니다.

### 6번 해설 - 시각화 (subplots) [빈칸채우기]

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(12, 5))
sns.countplot(data=data_merged, x='Customer_Segment', ax=axes[0])
sns.histplot(data=data_merged, x='Total_Purchase_Amount', hue='Churned', ax=axes[1])
plt.show()

> plt.subplots() 는 nrows/ncols 로 여러 축(Axes)을 한 번에 만듭니다. 정답: subplots

### 7번 해설 - 시각화 2 [코드작성]

In [ ]:
sns.boxplot(data=data_merged, x='Churned', y='Total_Purchase_Amount')
plt.show()

> sns.boxplot(x=, y=) / sns.jointplot(x=, y=) 형태로 그립니다.

### 8번 해설 - 이상치 처리 [빈칸채우기]

In [ ]:
q1 = data_merged['Total_Purchase_Amount'].quantile(0.25)
q3 = data_merged['Total_Purchase_Amount'].quantile(0.75)
iqr = q3 - q1
lower_fence = q1 - 2.0 * iqr
upper_fence = q3 + 2.0 * iqr
data_temp = data_merged.drop(data_merged[(data_merged['Total_Purchase_Amount'] > upper_fence) | (data_merged['Total_Purchase_Amount'] < lower_fence)].index)
data_temp = data_temp.drop(columns=['Customer_ID'])
data_temp = data_temp.reset_index(drop=True)

> DataFrame.drop() 은 행(기본 axis=0) 또는 열(axis=1)을 삭제합니다. 정답: drop

### 9번 해설 - 결측치 처리 [오류정정]

In [ ]:
fill_value = data_temp['Total_Purchase_Amount'].median()
data_na = data_temp.fillna({'Total_Purchase_Amount': fill_value})
data_na['Days_Since_Last_Purchase'] = data_na['Days_Since_Last_Purchase'].fillna(data_na['Days_Since_Last_Purchase'].median())

> [loop_control] 두 번째 결측치 채우기 단계에 불필요한 조건문(`if`)을 추가하여 실제 데이터 처리가 조건에 따라 건너뛰어질 수 있습니다. 이는 모든 결측치를 일관되게 처리해야 하는 데이터 정제 로직의 흐름을 잘못 제어합니다.

### 10번 해설 - 인코딩 [코드작성]

In [ ]:
label_cols = ['Region_Code']
dumm_cols = ['Customer_Segment']

from sklearn.preprocessing import LabelEncoder
data_preset = data_na.copy()
for col in label_cols:
    le = LabelEncoder()
    data_preset[col] = le.fit_transform(data_preset[col])

data_preset = pd.get_dummies(data=data_preset, columns=dumm_cols, drop_first=True)
data_preset['Churned'] = LabelEncoder().fit_transform(data_preset['Churned'])
data_preset.info()

> 저-카디널리티는 원-핫, 코드성 범주는 라벨 인코딩을 흔히 사용합니다. 타깃 컬럼도 XGBoost/LightGBM 등 일부 모델은 문자열 클래스를 그대로 받아들이지 않으므로, `Churned` 도 함께 LabelEncoder 로 0/1(다중클래스는 0..N-1) 정수로 인코딩합니다.

### 11번 해설 - 데이터 분리 [코드작성]

In [ ]:
from sklearn.model_selection import train_test_split

X = data_preset.drop(['Churned'], axis=1)
y = data_preset['Churned']

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=7, stratify=y)

> train_test_split(X, y, ...) 은 X_train, X_valid, y_train, y_valid 순서로 반환합니다.

### 12번 해설 - 스케일링 [오류정정]

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)

> [scope_reference] 이 코드는 훈련 데이터(X_train)에 대해 transform을 먼저 수행하고 검증 데이터(X_valid)에 대해 fit_transform을 수행하도록 순서를 뒤바꾸었습니다. 이는 스케일러가 훈련 데이터의 통계 정보를 기반으로 학습된 후, 검증 데이터에 적용되는 비논리적인 데이터 흐름 오류를 발생시킵니다.

### 13번 해설 - 스케일러 특성 [결과값예측]

In [ ]:
'평균(mean) 0, 표준편차(std) 1에 가까워집니다 (Z-score 표준화).'

> StandardScaler 의 정의에 따른 이론적 특성입니다.

### 14번 해설 - 머신러닝 기본 (fit-predict) [코드작성]

In [ ]:
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(random_state=7)
model.fit(X_train_scaled, y_train)
pred_y = model.predict(X_valid_scaled)
pred_y[:5]

> import → model = 클래스() → model.fit(X_train, y_train) → pred_y = model.predict(X_valid) 4줄 템플릿입니다.

### 15번 해설 - GridSearch 모델링 [코드작성]

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import StratifiedKFold
cv_obj = StratifiedKFold(n_splits=7, shuffle=True, random_state=7)

gs_a = GridSearchCV(DecisionTreeClassifier(random_state=7), {'max_depth':[3,5,7]}, cv=cv_obj, scoring='accuracy')
gs_a.fit(X_train_scaled, y_train)

gs_b = GridSearchCV(RandomForestClassifier(random_state=7), {'n_estimators':[50, 100, 200], 'max_depth':[3,5,7]}, cv=cv_obj, scoring='accuracy')
gs_b.fit(X_train_scaled, y_train)

> GridSearchCV(estimator, param_grid, cv=...).fit(X_train, y_train) 형태로 탐색합니다. RandomForestClassifier 는 sklearn 기본 앙상블 외에 XGBoost/LightGBM 계열일 수도 있습니다.

### 16번 해설 - GridSearch 결과 확인 [결과값예측]

In [ ]:
200

> 제시된 후보 중 GridSearch가 고를 수 있는 최댓값을 묻는 문항입니다.

### 17번 해설 - 변수중요도 [오류정정]

In [ ]:
fi = pd.DataFrame({'feature': X_train.columns, 'importance': gs_b.best_estimator_.feature_importances_})
fi = fi.sort_values('importance', ascending=True)[:15]
plt.barh(fi['feature'], fi['importance'])
plt.show()

> [operator_precedence] 연산자 우선순위/형변환 착각: 정수 나눗셈, 불필요한 나눗셈·곱셈 등으로 값이 왜곡됨

### 18번 해설 - 성능평가 [코드작성]

In [ ]:
from sklearn.metrics import roc_auc_score

proba_a = gs_a.best_estimator_.predict_proba(X_valid_scaled)[:, 1]
proba_b = gs_b.best_estimator_.predict_proba(X_valid_scaled)[:, 1]

a_score = roc_auc_score(y_valid, proba_a)
b_score = roc_auc_score(y_valid, proba_b)
print(a_score, b_score)

> roc_auc_score(정답, 예측확률) 형태이며 predict()가 아닌 predict_proba() 의 확률값을 사용합니다. 타깃이 이미 0/1 정수이므로 별도 이진화가 필요 없습니다.

### 19번 해설 - 모델 성능 비교 [결과값예측]

In [ ]:
'실행 결과에 따라 달라집니다: 값이 더 큰 쪽 이 더 우수한 모델입니다. a_score, b_score 를 직접 비교해서 판단하세요.'

> 정확도/F1/ROC-AUC 등은 높을수록, MAE/MSE 등 오차 지표는 낮을수록 좋은 성능입니다.

### 20번 해설 - 딥러닝 설계 [빈칸채우기]

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

model = Sequential([
    Dense(128, activation='elu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(64, activation='elu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(32, activation='elu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
cb_list = [EarlyStopping(monitor='val_loss', patience=14, restore_best_weights=True)]
cb_list.append(ModelCheckpoint('best_model.keras', monitor='val_loss', save_best_only=True))

> EarlyStopping(patience=N) 은 N번 연속 개선이 없으면 학습을 멈춥니다. 정답: patience

### 21번 해설 - 딥러닝 학습 [코드작성]

In [ ]:
history = model.fit(X_train_scaled, y_train, epochs=50, batch_size=16,
                    validation_data=(X_valid_scaled, y_valid), callbacks=cb_list)

> model.fit(X, y, epochs=, batch_size=, validation_data=, callbacks=) 형태입니다.

### 22번 해설 - 학습곡선 시각화 [코드작성]

In [ ]:
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Model accuracy')
plt.xlabel('epoch')
plt.ylabel('accuracy')
plt.legend(['train', 'val'], loc='upper left')
plt.show()

> history.history[지표] 로 epoch별 기록을 꺼내 plt.plot() + plt.legend(loc=...) 으로 그립니다.

### 23번 해설 - 저장된 모델 재사용 [코드작성]

In [ ]:
from tensorflow.keras.models import load_model

saved_model = load_model('best_model.keras')
reload_pred = saved_model.predict(X_valid_scaled)
reload_pred[:5]

> 학습 없이도 저장된 가중치를 불러와(load_model) 바로 predict() 할 수 있습니다.